In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import seaborn as sns
import string
import re
import warnings
import nltk
import numpy as np

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Read the dataset
url='/content/drive/MyDrive/fake_or_real_news.csv'
df=pd.read_csv(url)

In [ ]:
df.head()

In [ ]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
## Drop null values
news=df.dropna()

In [ ]:
news.isna().sum()

In [ ]:
print("Shape of the data set before drop the null values" + " " + str(df.shape))
print("Shape of the data set After drop the null values" + " " + str(news.shape))

In [ ]:
## Reset the index of the data set aftrer drop the null values
news.reset_index(inplace=True)

In [ ]:
news['text'][0]

In [ ]:
print("unreliable of the data set is"+ " "+str(news.label.value_counts()[1]))
print("reliable of the data set is"+ " "+str(news.label.value_counts()[0]))

In [ ]:
# True vs Non True News Distribution
sns.countplot(news.label)
plt.legend()
plt.title("Reliable vs Unreliable Distribution")

In [ ]:
news.label.value_counts()[0]

In [ ]:
#Describe about ham and Spam Mail Behavior in all data set
news.groupby ('label').describe()

In [ ]:
# Reliable of news
news[news.label == 'REAL'].describe()

In [ ]:
# UnReliable of news
news[news.label == 'FAKE'].describe()

In [ ]:
#Find word length of each news
news['Word_length']=news.text.apply(lambda x:len(x.split()))

In [ ]:
news.head()

In [ ]:
print("Reliable(Real news) news  length description : "+ "\n"+str(news[news['label']=='REAL'].describe()))
print("Unreliable(Fake) news  length description :"+ "\n"+str(news[news['label']=='FAKE'].describe()))

In [ ]:
sns.displot(news[news.label=='REAL'].Word_length,kind='kde')
sns.set_style(style='whitegrid')
plt.title("Reliable news length distribution")

In [ ]:
sns.displot(news[news.label=='FAKE'].Word_length,kind='kde')
sns.set_style(style='whitegrid')
plt.title("Uneliable news length distribution")

In [ ]:
pip install text-hammer

In [ ]:
import text_hammer as th
from tqdm.notebook import tqdm

def text_preprocessing(df, col_name):
    # فعال کردن نمایش tqdm برای pandas
    tqdm.pandas()

    # اعمال پردازش‌ها به ستون مشخص‌شده
    df[col_name] = df[col_name].progress_apply(lambda x: str(x).lower())
    df[col_name] = df[col_name].progress_apply(lambda x: th.cont_exp(x))  # تبدیل اختصارات
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_emails(x))  # حذف ایمیل‌ها
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_html_tags(x))  # حذف تگ‌های HTML
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_stopwords(x))  # حذف stopwords
    df[col_name] = df[col_name].progress_apply(lambda x: th.spelling_correction(x))  # اصلاح املایی
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_special_chars(x))  # حذف کاراکترهای خاص
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_accented_chars(x))  # حذف لهجه‌ها
    df[col_name] = df[col_name].progress_apply(lambda x: th.make_base(x))  # کاهش به ریشه کلمه

    return df

# فراخوانی تابع برای پردازش ستون "text" و ذخیره نتیجه در ستون "cleaned_text"
news['cleaned_text'] = text_preprocessing(news, "text")

In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Ensure you have the required NLTK data
nltk.download('stopwords')
nltk.download('punkt')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Remove non-alphabetical characters and convert to lowercase
    text = re.sub('[^a-zA-Z]', ' ', text).lower()

    # 2. Tokenize the text
    tokens = word_tokenize(text)

    # 3. Remove stopwords and apply stemming
    tokens = [ps.stem(word) for word in tokens if word not in stop_words]

    # 4. Join back into a single string
    return ' '.join(tokens)

# Apply the function to your corpus
corpus = [preprocess_text(text) for text in news['text']]


In [ ]:
#Convert corpus to array
import numpy as np
news['cleaned_text']=np.array(corpus)

In [ ]:
news['Word_length']=news.cleaned_text.apply(lambda x:len(x.split()))

In [ ]:
news.head()

In [ ]:
#Seperate Fake news
unreliable=list(news[news['label']=='FAKE']['cleaned_text'])

In [ ]:
#tokenize
from nltk.tokenize import word_tokenize
all_word_unreliable=[]
for sentence in unreliable:
    words=word_tokenize(sentence)
    for word in words:
        all_word_unreliable.append(word)

In [ ]:
from collections import Counter
counter_unreliable=Counter(all_word_unreliable)
print(counter_unreliable.most_common(20))
coun_unreliable=pd.DataFrame(counter_unreliable.most_common(20))
coun_unreliable.columns=['Common_Words','count']
coun_unreliable.style.background_gradient(cmap='Blues')
coun_unreliable

In [ ]:
##Plot Most Common words that appeared in mail
%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
fig = px.bar(coun_unreliable, x="count", y="Common_Words", title='Commmon Words in Fake News', orientation='h',
             width=700, height=700,color="Common_Words")
fig.show()

In [ ]:
from wordcloud import WordCloud,STOPWORDS
##Define function for Plot Word Cloud
def plot_wordcloud(text):
    stopwords = set(STOPWORDS)
    more_stopwords = {'u', "im",'mr'}
    stopwords = stopwords.union(more_stopwords)

    wordcloud = WordCloud(max_font_size=200,width=600, height=400).generate(str(text))
# Open a plot of the generated image.

    plt.figure( figsize=(20,10), facecolor='k')
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.title('WordCloud of spam mail')
    plt.show()

In [ ]:
plot_wordcloud(unreliable)

In [ ]:
## Seperate Reliable news message
reliable=list(news[news['label']=='REAL']['cleaned_text'])

In [ ]:
#Reliable  mail words
from nltk.tokenize import word_tokenize
all_word_reliable=[]
for sentence in reliable:
    words=word_tokenize(sentence)
    for word in words:
        all_word_reliable.append(word)

In [ ]:
from collections import Counter
counter_reliable=Counter(all_word_reliable)
print(counter_reliable.most_common(20))
coun_reliable=pd.DataFrame(counter_reliable.most_common(20))
coun_reliable.columns=['Common_Words','count']
coun_reliable.style.background_gradient(cmap='Blues')
coun_reliable

In [ ]:
##Plot Most Common words that appeared in mail
%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
fig = px.bar(coun_reliable, x="count", y="Common_Words", title='Commmon Words in Real News', orientation='h',
             width=700, height=700,color="Common_Words")
fig.show()

In [ ]:
#For ploting use created function
plot_wordcloud(reliable)

In [ ]:
#Applying Countvectorizer
#Creating the Bag Of Words model
from sklearn.feature_extraction.text import CountVectorizer,TfidfTransformer
cv=CountVectorizer(max_features=3500,ngram_range=(1,3))

In [ ]:
x=cv.fit_transform(corpus).toarray()

In [ ]:
print(x.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Bidirectional,Embedding,Dropout
from tensorflow.keras.layers import Dropout
from tensorflow.keras import activations

In [ ]:
x=news['cleaned_text']
y=news['label']

In [ ]:
import keras
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(x)

In [ ]:
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print("Vocab size : ",vocab_size)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# تبدیل مقادیر متنی به مقادیر عددی
label_encoder = LabelEncoder()
news['label'] = label_encoder.fit_transform(news['label'])

# تقسیم داده‌ها به مجموعه آموزش و تست
x_train, x_test, y_train, y_test = train_test_split(
    news['cleaned_text'],
    news['label'],
    test_size=0.2,
    stratify=news['label'],
    random_state=100
)


In [ ]:
from sklearn import model_selection
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=model_selection.train_test_split(news['cleaned_text'],news['label'].astype(int),test_size=0.2,stratify=news['label'],random_state=100)


In [ ]:
X_train_ =tokenizer.texts_to_sequences(x_train)
X_train_pad=pad_sequences(X_train_,maxlen=256,padding='post')
X_test_ = tokenizer.texts_to_sequences(x_test)
X_test_pad = pad_sequences(X_test_, maxlen = 256, padding = 'post')

In [ ]:
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Embedding, Dropout, Dense, Bidirectional, LSTM
import tensorflow as tf
from tensorflow.keras.metrics import Precision, Recall
# مشخصات مدل
EMBEDDING_DIM = 100
VOCAB_SIZE = 10000
INPUT_LENGTH = X_train_pad.shape[1]

# ورودی مدل
inputs = Input(shape=(INPUT_LENGTH,))

# لایه‌های مدل
x = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=INPUT_LENGTH)(inputs)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(256, return_sequences=True))(x)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(128, return_sequences=False))(x)
outputs = Dense(1, activation='sigmoid')(x)

# ساخت مدل
model = Model(inputs=inputs, outputs=outputs)

# کامپایل مدل
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adamax(),
    metrics=['acc', Precision(name="Precision"), Recall(name="Recall")]
)

# نمایش ساختار مدل
model.summary()


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dropout, Dense, Bidirectional, LSTM
import tensorflow as tf

EMBEDDING_DIM = 100
model = Sequential()
model.add(Embedding(input_dim=10000,  # اندازه واژگان
                    output_dim=EMBEDDING_DIM,
                    input_length=X_train_pad.shape[1]))  # طول ورودی
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(256, return_sequences=True)))
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dense(1, activation='sigmoid'))

# کامپایل مدل
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adamax(),
    metrics=['accuracy']
)

# نمایش ساختار مدل
model.summary()


In [ ]:
history_emd= model.fit(X_train_pad,y_train, epochs = 30, batch_size = 200, validation_data=(X_test_pad, y_test),verbose = 1)


In [ ]:
plt.plot(history_emd.history['acc'],c='b',label='train accuracy')
plt.plot(history_emd.history['val_acc'],c='r',label='validation accuracy')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import numpy as np

# پیش‌بینی احتمالات
pred_probs = model.predict(X_test_pad)

# تبدیل احتمالات به کلاس‌ها
pred = (pred_probs > 0.5).astype("int32")


In [ ]:
# plotting graph for confusion matrix
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5,5))
sns.heatmap(cm, annot=True, fmt='', cbar=False, linewidths=2,
            xticklabels = ['Fake','Real'], yticklabels = ['Fake','Real'])
plt.title('confusion matrix')
plt.xlabel('Predicted', color='navy', fontsize=15)
plt.ylabel('Actual', color='navy', fontsize=15);

In [ ]:
# accuracy of the model we built
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

print("Model Accuracy : ", accuracy)
print("Model Precision: ", precision)
print("Model Recall   : ", recall)
print("Model F1 Score : ", f1)